In [7]:
import pandas as pd 
import numpy as np
import joblib

import warnings 
warnings.filterwarnings('ignore')

In [8]:
df_new = pd.read_excel('Prediction_Data.xlsx', sheet_name='Sheet2')

print(f"Loaded {df_new.shape[0]} new joinees for prediction.")
display(df_new.head())

Loaded 411 new joinees for prediction.


,Customer_ID,Gender,Age,Married,State,Number_of_Referrals,Tenure_in_Months,Value_Deal,Phone_Service,Multiple_Lines,...,Payment_Method,Monthly_Charge,Total_Charges,Total_Refunds,Total_Extra_Data_Charges,Total_Long_Distance_Charges,Total_Revenue,Customer_Status,Churn_Category,Churn_Reason
0,93520-GUJ,Female,67,No,Gujarat,13,19,Deal 5,Yes,Yes,...,Bank Withdrawal,72.10,72.1,0.0,0,7.77,79.87,Joined,Not Applicable,Not Applicabe
1,57256-BIH,Female,18,No,Bihar,9,7,NaN,Yes,No,...,Credit Card,19.85,57.2,0.0,0,9.36,66.56,Joined,Not Applicable,Not Applicabe
2,72357-MAD,Female,53,No,Madhya Pradesh,14,12,Deal 5,Yes,No,...,Credit Card,44.30,44.3,0.0,0,42.95,87.25,Joined,Not Applicable,Not Applicabe
3,66612-KAR,Female,58,Yes,Karnataka,11,18,NaN,Yes,No,...,Credit Card,19.95,58.0,0.0,0,8.07,66.07,Joined,Not Applicable,Not Applicabe
4,22119-WES,Male,31,Yes,West Bengal,5,5,NaN,Yes,No,...,Credit Card,20.05,33.7,0.0,0,3.62,37.32,Joined,Not Applicable,Not Applicabe


In [9]:
model = joblib.load('best_churn_model.joblib')
scaler = joblib.load('scaler.joblib')
training_columns = joblib.load('training_columns.joblib')


In [10]:
df_new.isnull().sum()

Customer_ID                      0
Gender                           0
Age                              0
Married                          0
State                            0
Number_of_Referrals              0
Tenure_in_Months                 0
Value_Deal                     251
Phone_Service                    0
Multiple_Lines                   0
Internet_Service                 0
Internet_Type                  167
Online_Security                  0
Online_Backup                    0
Device_Protection_Plan           0
Premium_Support                  0
Streaming_TV                     0
Streaming_Movies                 0
Streaming_Music                  0
Unlimited_Data                   0
Contract                         0
Paperless_Billing                0
Payment_Method                   0
Monthly_Charge                   0
Total_Charges                    0
Total_Refunds                    0
Total_Extra_Data_Charges         0
Total_Long_Distance_Charges      0
Total_Revenue       

In [11]:
# Safe copy
df_final = df_new.copy()

# Droping irrelevant columns 
columns_to_drop = ['Customer_ID', 'Customer_Status', 'Churn_Category', 'Churn_Reason']

X_new = df_new.drop(columns=[col for col in columns_to_drop if col in df_new.columns])


# Handle missing Values

X_new['Value_Deal'].fillna('None',inplace=True)
X_new['Internet_Type'].fillna(X_new['Internet_Type'].mode()[0],inplace=True)


# One-Hot Encode
X_new_encoded = pd.get_dummies(X_new, drop_first = True)
X_new_encoded = X_new_encoded.reindex(columns=training_columns, fill_value=0)


In [12]:
# Scaling
X_new_scaled = scaler.transform(X_new_encoded)

In [14]:
# Predictions
hard_predictions = model.predict(X_new_scaled)
df_final['Predicted_Class'] = hard_predictions

# Map it to readable text for Power BI
df_final['Prediction_Label'] = df_final['Predicted_Class'].map({1: 'Predicted Churner', 0: 'Predicted Safe'})

churn_probabilities = model.predict_proba(X_new_scaled)[:, 1]

df_final['Churn_Probability_%'] = np.round(churn_probabilities*100,2)

def assign_risk_level(prob):
    if prob >= 70:
        return 'High Risk'
    elif prob >= 40:
        return 'Medium Risk'
    else:
        return 'Low Risk'

df_final['Risk_Level'] = df_final['Churn_Probability_%'].apply(assign_risk_level)

display(df_final[['Customer_ID','Prediction_Label', 'Churn_Probability_%','Risk_Level']].head(10))



,Customer_ID,Prediction_Label,Churn_Probability_%,Risk_Level
0,93520-GUJ,Predicted Churner,75.00,High Risk
1,57256-BIH,Predicted Safe,26.21,Low Risk
2,72357-MAD,Predicted Churner,75.75,High Risk
3,66612-KAR,Predicted Safe,25.77,Low Risk
4,22119-WES,Predicted Safe,18.38,Low Risk
5,79168-UTT,Predicted Churner,64.14,Medium Risk
6,66712-GUJ,Predicted Churner,88.97,High Risk
7,53815-BIH,Predicted Safe,30.57,Low Risk
8,34110-MAH,Predicted Churner,71.06,High Risk
9,12056-WES,Predicted Churner,80.37,High Risk


In [15]:
# Final Predicted File

df_final.to_csv('New_Joinees_Risk_Predictions(Logistic Regression).csv',index=False)

In [16]:
(df_final['Risk_Level'].value_counts()/len(df_final))*100

Risk_Level
High Risk      36.009732
Low Risk       34.793187
Medium Risk    29.197080
Name: count, dtype: float64

In [17]:
len(df_final)

411